# Optimize three complex launch coefficients for LP01 preservation

Propagate all three coupled modes at the original fixed total peak powers, **100 kW** and **81.943 kW**. Keep the equal-amplitude, zero-phase starts and optimize the same controls:

$$a_m=\frac{c_m e^{i\phi_m}}{\sqrt{\sum_jc_j^2}},\qquad P_m/P=|a_m|^2,$$

with $c_m\ge0$, $\sum_m c_m=100$, $\phi_1=0$, and two relative phases modulo $2\pi$. Five stored coordinates describe four independent physical controls.

Score **only the original LP01 input and nonlinear LP01 output**:

$$F_{\rm IO}=\frac{\left|\int A_1^*(0,t)A_{{\rm NL},1}(L,t)\,dt\right|^2}{\left(\int |A_1(0,t)|^2dt\right)\left(\int |A_{{\rm NL},1}(L,t)|^2dt\right)},\qquad D_{\rm IO}=1-F_{\rm IO}.$$

The numerator and both energy factors include **mode 1 only**. Modes 2 and 3 remain in nonlinear propagation and may affect LP01 through coupling. The same time grid is used without time shifting or phase alignment; common phase and scale cancel naturally.

Require LP01 output photons to stay above **50% of the equal-amplitude fine-grid baseline at the same total power**. An absent LP01 reference or output receives the worst score, $D_{\rm IO}=1$, and is infeasible rather than being treated as perfect preservation. Linear propagation is a baseline/selected plotting diagnostic only.

Run all cells with **GNLSE (Julia)**. The original notebooks and preceding run data are preserved.

In [ ]:
using Pkg
workspace = let dir = abspath(@__DIR__)
    while !isfile(joinpath(dir, "notebook_helpers", "run_library.py"))
        dirname(dir) == dir && error("Open this notebook inside the Nonlinear Optics workspace.")
        dir = dirname(dir)
    end
    dir
end
ROOT = get(ENV, "DIO_PROJECT_ROOT", joinpath(workspace, "FastDifferentiableMMNLO"))
Pkg.activate(ROOT)
using PulsePropagation, FFTW, LinearAlgebra, Printf, TOML, Dates, SHA, NPZ
using PyPlot

BLAS.set_num_threads(1)
FFTW.set_num_threads(1)
run_dir = haskey(ENV, "DIO_RUN_DIR") ? ENV["DIO_RUN_DIR"] :
    mktempdir(mkpath(joinpath(workspace, "output", "organized", "runs", "Modal amplitudes and relative phases", "Gradient descent")); prefix=Dates.format(now(), "yyyy-mm-dd") * "_dio_phase_amplitude_optimization_", cleanup=false)
mkpath(run_dir)
println("Results: ", run_dir)
plot_settings_path = get(ENV, "DIO_PLOT_SETTINGS", joinpath(workspace, "notebook_helpers", "julia", "plot_settings.toml"))
beam_settings = TOML.parsefile(plot_settings_path)
beam_color_min = Float64(beam_settings["beam_intensity_min_W_per_um2"])
beam_color_max = Float64(beam_settings["beam_intensity_max_W_per_um2"])
beam_colormap = String(beam_settings["colormap"])
beam_color_ticks = collect(range(beam_color_min, beam_color_max; length=6))

# Load the launch, objective, descent, and plotting functions used below.
include(joinpath(workspace, "notebook_helpers", "julia", "Modal amplitudes and relative phases", "shared.jl"))
include(joinpath(workspace, "notebook_helpers", "julia", "Modal amplitudes and relative phases", "dio_phase_amplitude_optimization.jl"))

## 1. Preserve the launch controls and bounded search

Use $q=(c_1/100,c_2/100,c_3/100,\phi_2,\phi_3)$, with radians for phase.
Project amplitudes onto the sum-one simplex and wrap phases into $[-\pi,\pi)$.
Keep the learning rate 0.10, coefficient cap 10, phase cap 0.10 rad, amplitude finite-difference step 0.5, and phase finite-difference step 0.01 rad.

Each power case has **at most 20 accepted iterations and 200 new optimization evaluations**, including its starting point and finite-difference probes. A complete gradient normally costs ten nonlinear solves. The run stops at its actual budget or line-search condition.

Validation consists of one **1 mm baseline** and one **1 mm selected launch** per power. The search uses the original 5 mm step and 2048 samples. No dense/CP comparison, larger time grid, repeat gradient, perturbation campaign, or multistart is performed.

In [ ]:
power_cases_W = [100_000.0, 81_943.0]
start_coefficients = fill(100.0/3, 3)
start_phases_rad = zeros(3)
phase_fd_step = 0.01                # radians
max_phase_change_rad = 0.10         # local angular increment, about 5.7 degrees
coefficient_bounds = (0.0, 100.0)
fd_step = 0.5                       # amplitude-coefficient units
learning_rate = 0.10                # update q = (c/100, phi2, phi3)
max_iterations = 20
max_optimizer_evaluations = 200     # per fixed-power case
max_check_evaluations = 2           # one fine baseline + one fine selected launch
max_coefficient_change = 10.0
gradient_tolerance = 1e-4           # amplitude projected gradient and phase derivatives in q
minimum_brightness_fraction = 0.50
search_dz = 0.005                   # 5 mm
validation_dz = 0.001               # 1 mm
dio_tolerance = 2e-4
minimum_decrease = 1e-10

settings = Dict(
    "control"=>"three amplitudes sum(c)=100 and two relative phases; phi1=0",
    "objective"=>"LP01 original input/nonlinear output complex-field D_IO; mode-1-only norms; no alignment",
    "objective_modes"=>[1], "objective_filter"=>"none",
    "empty_LP01_policy"=>"F_IO=0, D_IO=1, infeasible if input or output LP01 norm is zero",
    "propagated_modes"=>[1,2,3],
    "power_cases_W"=>power_cases_W, "start_coefficients"=>start_coefficients,
    "coefficient_bounds"=>collect(coefficient_bounds),
    "start_phases_rad"=>start_phases_rad, "phase_range_rad"=>[-Float64(pi),Float64(pi)],
    "phase_range_convention"=>"periodic [-pi,pi); phi1=0; absent-mode phase is immaterial",
    "phase_fd_step_rad"=>phase_fd_step,"max_phase_change_rad"=>max_phase_change_rad,
    "gradient_coordinates"=>["c1/100","c2/100","c3/100","phi2_rad","phi3_rad"],
    "finite_difference_step_coefficient"=>fd_step,
    "learning_rate_q"=>learning_rate, "max_iterations"=>max_iterations,
    "max_optimizer_evaluations_per_power"=>max_optimizer_evaluations,
    "max_check_evaluations_per_power"=>max_check_evaluations,
    "max_coefficient_change"=>max_coefficient_change,
    "gradient_tolerance_q"=>gradient_tolerance,
    "minimum_brightness_fraction"=>minimum_brightness_fraction,
    "brightness_definition"=>"LP01 unfiltered photons relative to equal-amplitude three-mode fine baseline at the same total power",
    "search_dz_m"=>search_dz, "validation_dz_m"=>validation_dz,
    "dio_tolerance"=>dio_tolerance,
    "armijo_c1"=>1e-4,"max_backtracking_halvings"=>12,
    "minimum_D_IO_decrease"=>minimum_decrease,
    "Nt"=>2048, "time_window_ps"=>10.0,
    "pulse_fwhm_ps"=>0.2, "fiber_length_m"=>2.86, "carrier_m"=>1379.2e-9,
    "core_radius_m"=>52.5e-6, "NA"=>0.1, "num_modes"=>3,
    "linear_reference"=>"baseline/selected plot diagnostic only; n2 = 0",
    "cp_target_error"=>1e-3, "cp_seed"=>Int(0x4d4d474e),
    "julia_version"=>string(VERSION),
    "manifest_sha256"=>bytes2hex(sha256(read(joinpath(ROOT, "Manifest.toml")))),
)
for P in power_cases_W
    mkpath(case_dir(P))
end
write_toml(joinpath(run_dir, "settings.toml"), settings)
println("Fixed powers (kW): ", power_cases_W./1000)
println("Initial coefficients: ", start_coefficients)
println("Initial phases (rad): ", start_phases_rad)

## 2. Build the fixed three-mode fiber once

Mode 1 is LP01; modes 2 and 3 are the cosine and sine LP11 orientations. Reuse the same mode profiles, dispersion, and CP overlap tensor for both power cases. All three modes propagate and interact; the objective and brightness constraint use LP01 only.

In [ ]:
properties = compute_fiber_properties(
    StepIndex(settings["carrier_m"], settings["core_radius_m"], settings["NA"]),
    Silica(), settings["num_modes"];
    polarization=:scalar, mode_basis=:LP, beta_order=5)
S_cp = cp_compress(properties.S; error=settings["cp_target_error"], seed=settings["cp_seed"])
context_cache = Dict{Tuple{Int,Symbol},Any}()
main_context = simulation_context(settings["Nt"])
time_grid, domain = main_context.grid, main_context.domain
mode_names = ("LP01", "LP11-cos", "LP11-sin")
@printf("CP rank: %d; relative tensor error: %.4g\n", cp_rank(S_cp), S_cp.relative_error)
println("Mode labels: ", properties.modes.labels)

## 3. Score LP01 input–output preservation

`dio_at(c, P; phases_rad=phi)` normalizes the full three-mode launch and performs one nonlinear propagation. Its objective calls `field_fidelity(input, nonlinear; selected_modes=1:1)`: only LP01 enters the overlap and both norm factors. Uniform time-bin factors cancel in the normalized ratio.

Cache keys retain the full launch controls. Each new evaluation checks total launch normalization and finite fields. An empty LP01 reference or output returns F_IO = 0 with an explicit invalid-objective flag; line search rejects that trial without aborting the run. The LP01 photon count defines the brightness constraint.

In [ ]:
evaluation_cache = Dict{Any,NamedTuple}()
field_cache = Dict{Int,NamedTuple}()
evaluations, failures = NamedTuple[], NamedTuple[]
attempt_id = Ref(0)
used = Dict(P=>Dict(:optimization=>0, :validation=>0) for P in power_cases_W)

# Targeted algebra checks only; these require no propagation.
let
    u=project_simplex([-2.0,3.0,0.4])
    @assert all(u .>= 0) && isapprox(sum(u),1.0;atol=1e-14)
    a=zeros(ComplexF64,4,3,1);a[1,1,1]=1;a[1,2,1]=1
    other_mode_change=copy(a);other_mode_change[2,2,1]=4;other_mode_change[3,3,1]=3im
    @assert field_fidelity(a,other_mode_change;selected_modes=1:1) == 1
    distorted=copy(a);distorted[2,1,1]=1
    @assert field_fidelity(a,distorted;selected_modes=1:1) < 1
    @assert isapprox(field_fidelity(a,2exp(0.7im).*a;selected_modes=1:1),1;atol=1e-12)
    empty_lp01=copy(a);empty_lp01[:,1,:].=0
    @assert field_fidelity(empty_lp01,a;selected_modes=1:1) == 0
    @assert field_fidelity(a,empty_lp01;selected_modes=1:1) == 0
    @assert canonical_phases([0.6,1.3,0.2]) ≈ [0.0,0.7,-0.4]
end
println("LP01-only overlap, temporal distortion, phase/scale invariance, empty-mode, and simplex checks passed.")

## 4. Estimate amplitude and phase derivatives

Perturb each raw amplitude by ±0.5 and each relative phase by ±0.01 rad. Each probe normalizes the launch at fixed total power and evaluates D_IO. Multiply amplitude derivatives by 100 to obtain $dD/d(c_m/100)$; phase derivatives are per radian. Use one-sided second-order amplitude differences near bounds. Phase probes remain central across the periodic boundary.

In [ ]:
# The setup cell loads these functions from the study helper file.

## 5. Record the equal-amplitude, zero-phase starts

The starting search-grid value counts toward optimization. One 1 mm baseline establishes the LP01 brightness floor and records the search/fine D_IO difference. A discrepancy is reported without changing numerical settings or consuming extra probes.

In [ ]:
preflights=Dict{Float64,Any}()
brightness_floors=Dict{Float64,Float64}()
for P in power_cases_W
    baseline=dio_at(start_coefficients,P;phases_rad=start_phases_rad,phase=:optimization)
    fine=dio_at(start_coefficients,P;phases_rad=start_phases_rad,dz=validation_dz)
    step_error=abs(baseline.D_IO-fine.D_IO)
    fine.photons_lp01 > 0 || error("LP01 fine baseline must have positive brightness.")
    brightness_floors[P]=minimum_brightness_fraction*fine.photons_lp01
    preflights[P]=(;baseline,fine,step_error)
    write_toml(joinpath(case_dir(P),"preflight.toml"),Dict(
        "power_W"=>P,"baseline_D_IO_search"=>baseline.D_IO,
        "baseline_D_IO_fine"=>fine.D_IO,"step_error"=>step_error,
        "LP01_brightness_floor"=>brightness_floors[P]))
    @printf("%.3f kW: baseline D_IO %.8f (5 mm), %.8f (1 mm); difference %.3g\n",
        P/1000,baseline.D_IO,fine.D_IO,step_error)
end

## 6. Move amplitudes and phases together

Estimate all five derivatives, project the proposed amplitude step, and use the
negative phase gradient for the local angular step. Scale the whole direction until
the amplitude and phase caps are satisfied, then apply the existing Armijo backtracking.
The score must decrease and the LP01 brightness floor must be met.

Use the **unwrapped local angular increments** in the gradient-dot-step term. Wrap
only the stored phase coordinates. This prevents a harmless crossing of ±π from
looking like a large physical jump. Stop after 20 accepted steps, 200 optimization
evaluations, a small projected/tangent gradient, or a failed line search.

In [ ]:
optimizations=Dict(P=>optimize_launch(P) for P in power_cases_W)

## 7. Validate the selected launch once on the 1 mm grid

Compare each selected D_IO with its matched fine baseline and search-grid value. Check finite feasible controls, decreasing accepted scores, bounded moves, and the 50% LP01 photon floor. Record actual termination and budget use. These checks do not establish fine-grid stationarity or a global minimum.

In [ ]:
validations=Dict{Float64,Any}()
summaries=Dict{Float64,Any}()
comparison_rows=NamedTuple[]
for P in power_cases_W
    opt=optimizations[P];base_fine=preflights[P].fine
    c,p=coefficients(opt.best),phase_values(opt.best)
    best_fine=dio_at(c,P;phases_rad=p,dz=validation_dz)
    dz_error=max(preflights[P].step_error,abs(opt.best.D_IO-best_fine.D_IO))
    history_ok=all(r->all(x->0<=x<=100,coefficients(r)) &&
        isapprox(sum(coefficients(r)),100;atol=1e-10) && phase_values(r)[1]==0 &&
        all(x->-pi<=x<pi,phase_values(r)),opt.history) &&
        all(diff([r.D_IO for r in opt.history]).<0) &&
        all(r->r.max_coefficient_step<=max_coefficient_change+1e-10 &&
            r.max_phase_step_rad<=max_phase_change_rad+1e-12,opt.history)
    checks=Dict("accepted_path_constraints_and_decrease"=>history_ok,
        "accepted_path_LP01_brightness"=>all(r->r.brightness_ratio>=minimum_brightness_fraction,opt.history),
        "step_size_agreement"=>dz_error<=dio_tolerance,
        "LP01_objective_valid"=>best_fine.objective_valid && base_fine.objective_valid && all(r->r.objective_valid,opt.history),
        "fine_LP01_brightness"=>best_fine.photons_lp01>=brightness_floors[P])
    check_status=Dict(k=>(v ? "pass" : "fail") for (k,v) in checks)
    for omitted in ("cp_dense_comparison","larger_time_grid","fine_gradient",
                    "repeat_perturbations","phase_diagnostic_probes","multistart")
        check_status[omitted]="not_run"
    end
    improvement=base_fine.D_IO-best_fine.D_IO
    summary=Dict(
        "power_W"=>P,"objective"=>"LP01 original input/nonlinear output; mode-1-only norms; no alignment", "objective_modes"=>[1], "propagated_modes"=>[1,2,3],
        "stop_reason"=>opt.status,"accepted_steps"=>length(opt.history)-1,
        "selected_coefficients"=>c,"selected_phases_rad"=>p,"selected_phases_degrees"=>rad2deg.(p),
        "selected_input_power_fractions"=>c.^2 ./ sum(abs2,c),
        "baseline_D_IO_search"=>preflights[P].baseline.D_IO,"selected_D_IO_search"=>opt.best.D_IO,
        "baseline_D_IO_fine"=>base_fine.D_IO,"selected_D_IO_fine"=>best_fine.D_IO,
        "absolute_D_IO_improvement"=>improvement,"fractional_D_IO_reduction"=>improvement/base_fine.D_IO,
        "brightness_ratio"=>best_fine.photons_lp01/base_fine.photons_lp01,
        "brightness_definition"=>"LP01 output photons relative to matched equal-amplitude fine baseline",
        "max_step_size_D_IO_difference"=>dz_error,"essential_checks_passed"=>all(values(checks)),
        "fine_grid_improvement"=>improvement>0,"fine_grid_stationarity_checked"=>false,
        "global_minimum_established"=>false,
        "new_optimization_evaluations"=>used[P][:optimization],"new_check_evaluations"=>used[P][:validation],
        "failed_evaluations"=>count(r->r.power_W==P,failures),"checks"=>check_status)
    summaries[P]=summary
    validations[P]=(;base_fine,best_fine)
    for (label,row) in (("equal_zero_phase_baseline",base_fine),("phase_amplitude_selected",best_fine))
        push!(comparison_rows,comparison_row(label,row,base_fine))
    end
    write_toml(joinpath(case_dir(P),"summary.toml"),summary)
    save_rows(joinpath(case_dir(P),"comparisons.csv"),filter(r->r.power_W==P,comparison_rows))
    @printf("%.3f kW: fine D_IO %.8f -> %.8f; essential checks=%s; stop=%s\n",
        P/1000,base_fine.D_IO,best_fine.D_IO,string(all(values(checks))),opt.status)
    println("  coefficients ",coeff_label(c),"; phases ",phase_label(p))
    @printf("  optimization %d/%d; fine checks %d/%d\n",
        used[P][:optimization],max_optimizer_evaluations,used[P][:validation],max_check_evaluations)
end
save_rows(joinpath(run_dir,"comparisons.csv"),comparison_rows)
write_toml(joinpath(run_dir,"summary.toml"),Dict(power_label(P)=>summaries[P] for P in power_cases_W))

# Four diagnostic linear solves only, after objective search and validation.
linear_diagnostic_rows=NamedTuple[]
for P in power_cases_W, row in (validations[P].base_fine,validations[P].best_fine)
    saved=field_cache[row.id]
    linear_run=@timed solve_mmgnlse(saved.input,main_context.linear,row.dz)
    linear=get_temporal_field(linear_run.value;z=:final)
    field_cache[row.id]=merge(saved,(;linear=copy(linear)))
    push!(linear_diagnostic_rows,(;power_W=P,evaluation_id=row.id,seconds=linear_run.time))
end
save_rows(joinpath(run_dir,"linear_diagnostics.csv"),linear_diagnostic_rows)

## 8. Follow the amplitudes, modal powers, and phases

The first figure shows D_IO, the three amplitudes, and their modal power fractions.
The phase figure shows both relative phases in degrees; the calculation uses radians.
The mixture triangle projects the saved evaluations onto modal power fractions, so
points at the same location can have different phases and different D_IO. It is not
the full four-dimensional landscape or a fitted surface.

In [ ]:
mode_colors=("#366b9e","#cf7c32","#439576")
mode_styles=("-","-","--")
mode_markers=("o","s","none")
fig,axes=subplots(2,3;figsize=(16,8.4),constrained_layout=true)
for (row_index,P) in enumerate(power_cases_W)
    opt=optimizations[P];h=opt.history
    iterations=[r.iteration for r in h]
    axes[row_index,1].plot(iterations,[r.D_IO for r in h],"o-";
        color=mode_colors[1],markersize=4,label="Accepted path: 5 mm")
    axes[row_index,1].scatter([0,iterations[end]],
        [validations[P].base_fine.D_IO,validations[P].best_fine.D_IO];
        color="black",marker="x",s=60,label="Fine check: 1 mm",zorder=5)
    axes[row_index,1].set_ylabel("LP01 D_IO")
    axes[row_index,1].set_title(@sprintf("%.3f kW | %s",P/1000,replace(opt.status,"_"=>" "));fontsize=10)
    axes[row_index,1].legend(;fontsize=8)
    for m in 1:3
        axes[row_index,2].plot(iterations,[coefficients(r)[m] for r in h];
            color=mode_colors[m],linestyle=mode_styles[m],marker=mode_markers[m],markersize=3,label=mode_names[m])
        axes[row_index,3].plot(iterations,[100coefficients(r)[m]^2/sum(abs2,coefficients(r)) for r in h];
            color=mode_colors[m],linestyle=mode_styles[m],marker=mode_markers[m],markersize=3,label=mode_names[m])
    end
    axes[row_index,2].set_ylabel("Amplitude coefficient (sum = 100)")
    axes[row_index,3].set_ylabel("Input modal power (%)")
    for j in 2:3
        axes[row_index,j].set_ylim(-2,102)
        axes[row_index,j].legend(;fontsize=8)
    end
    for j in 1:3
        axes[row_index,j].set_xlabel("Accepted iteration")
        tidy_axis(axes[row_index,j])
    end
end
fig.suptitle("Complex launch optimization: amplitude and power histories")
finish_figure(fig,joinpath(run_dir,"optimization_history"))
fig,axes=subplots(1,2;figsize=(13,5.7),constrained_layout=true)
for (j,P) in enumerate(power_cases_W)
    ax=axes[j]
    rows=filter(r->r.power_W==P && r.dz==search_dz && r.nt==settings["Nt"] && r.representation=="cp",evaluations)
    xy=[triangle_xy(coefficients(r)) for r in rows]
    dots=ax.scatter(first.(xy),last.(xy);c=[r.D_IO for r in rows],cmap="viridis",s=18,alpha=0.8)
    path=[triangle_xy(coefficients(r)) for r in optimizations[P].history]
    ax.plot(first.(path),last.(path),"o-";color="#ca584b",markersize=3,label="Accepted steps",zorder=5)
    ax.scatter([first(path)[1]],[first(path)[2]];marker="o",s=85,
        facecolors="white",edgecolors="black",label="Equal-amplitude start",zorder=6)
    ax.scatter([last(path)[1]],[last(path)[2]];marker="*",s=155,
        color="#e79b36",edgecolors="black",label="Selected",zorder=7)
    ax.plot([0,1,0.5,0],[0,0,sqrt(3)/2,0];color="#888888",linewidth=1)
    ax.text(-0.03,-0.035,"LP01";ha="center",fontsize=10)
    ax.text(1.03,-0.035,"LP11-cos";ha="center",fontsize=10)
    ax.text(0.5,sqrt(3)/2+0.025,"LP11-sin";ha="center",fontsize=10)
    ax.set_aspect("equal");ax.set_xlim(-0.13,1.13);ax.set_ylim(-0.12,1.0)
    ax.axis("off")
    ax.set_title(@sprintf("%.3f kW: mixture by input power fraction",P/1000))
    ax.legend(;loc="upper left",fontsize=7.5)
    fig.colorbar(dots;ax,shrink=0.65,label="Sampled LP01 D_IO (5 mm)")
end
finish_figure(fig,joinpath(run_dir,"sampled_mixture_triangles"))

fig,axes=subplots(1,2;figsize=(12.5,4.5),constrained_layout=true)
for (j,P) in enumerate(power_cases_W)
    h=optimizations[P].history
    for m in 2:3
        axes[j].plot([r.iteration for r in h],[rad2deg(phase_values(r)[m]) for r in h];
            color=mode_colors[m],linestyle=mode_styles[m],marker=mode_markers[m],markersize=3,
            label=mode_names[m]*" relative to LP01")
    end
    axes[j].axhline(0;color="#aaaaaa",linewidth=0.8)
    axes[j].set_xlabel("Accepted iteration");axes[j].set_ylabel("Relative launch phase (degrees)")
    axes[j].set_title(@sprintf("%.3f kW; phi1 = 0",P/1000))
    axes[j].legend(;fontsize=8);tidy_axis(axes[j])
end
fig.suptitle("Phase coordinates are free; a symmetric start can still have small phase gradients")
finish_figure(fig,joinpath(run_dir,"phase_history"))
fig,axes=subplots(1,2;figsize=(12.5,4.5),constrained_layout=true)
for (j,P) in enumerate(power_cases_W)
    h=optimizations[P].history[2:end]
    if !isempty(h)
        axes[j].semilogy([r.iteration-1 for r in h],[max(r.previous_projected_gradient_norm,eps()) for r in h],"o-";
            color=mode_colors[1],markersize=3,label="Projected/tangent gradient norm")
        for (m,field) in ((2,:previous_gradient_phi2),(3,:previous_gradient_phi3))
            axes[j].semilogy([r.iteration-1 for r in h],[max(abs(getproperty(r,field)),eps()) for r in h];
                color=mode_colors[m],linestyle=mode_styles[m],label="|phase derivative| "*mode_names[m])
        end
    end
    axes[j].set_xlabel("Iteration before accepted move")
    axes[j].set_ylabel("Gradient magnitude")
    axes[j].set_title(@sprintf("%.3f kW: gradients used for accepted moves",P/1000);fontsize=10)
    axes[j].legend(;fontsize=8);tidy_axis(axes[j])
end
finish_figure(fig,joinpath(run_dir,"gradient_history"))

## 9. Compare baseline and selected fields

LP01 D_IO bars use the 1 mm nonlinear solves. Time traces and wavelength spectra show the original input, diagnostic linear output, and nonlinear output. Spectral ordinate values remain power per sampled frequency bin; the horizontal coordinate is wavelength in nm, with no detuning plots.

In [ ]:
fig,axes=subplots(1,2;figsize=(11,4.7),constrained_layout=true)
for (j,P) in enumerate(power_cases_W)
    v=validations[P];rows=(v.base_fine,v.best_fine)
    axes[j].bar(1:2,[r.D_IO for r in rows];color=["#a8b6c6","#366b9e"])
    for k in 1:2
        axes[j].text(k,rows[k].D_IO+0.008,@sprintf("%.5f",rows[k].D_IO);ha="center",fontsize=9)
    end
    axes[j].set_xticks(1:2,["Equal amplitudes\nzero phases","Complex launch\nselected"])
    axes[j].set_ylim(0,min(1.08,1.17maximum(r.D_IO for r in rows)+0.02))
    axes[j].set_ylabel("LP01 D_IO (1 mm)")
    axes[j].set_title(@sprintf("%.3f kW",P/1000));tidy_axis(axes[j])
end
finish_figure(fig,joinpath(run_dir,"baseline_selected_comparisons"))

times_ps=collect(time_axis(time_grid))
wavelengths_nm=collect(wavelength_axis(time_grid,properties.omega0)).*1e9
wavelength_indices=findall(isfinite.(wavelengths_nm).&(wavelengths_nm.>0))
wavelength_indices=wavelength_indices[sortperm(wavelengths_nm[wavelength_indices])]
curve_labels=("Input","Linear reference","Nonlinear output")
curve_colors=("#777777","#cf7c32","#366b9e")
curve_styles=(":","--","-")
for P in power_cases_W
    plot_modal_output("baseline",validations[P].base_fine)
    plot_modal_output("selected",validations[P].best_fine)
end

## 10. See the spatial modes and where the energy goes

The spatial mode shapes are fixed by the fiber. Their signed field amplitudes are
shown below. The bars show how the total pulse energy is distributed among modes;
these fractions can change during nonlinear propagation.

In [ ]:
x,y=properties.modes.x,properties.modes.y
modes=properties.modes.fields # y, x, mode
let fig=figure(;figsize=(15,4.8),constrained_layout=true)
    for m in 1:3
        ax=subplot(1,3,m)
        profile=real.(modes[:,:,m]);profile ./= maximum(abs,profile)
        mesh=ax.pcolormesh(x.*1e6,y.*1e6,profile;cmap="seismic",vmin=-1,vmax=1,
            shading="auto",rasterized=true)
        theta=range(0,2pi;length=181)
        ax.plot(settings["core_radius_m"].*1e6.*cos.(theta),settings["core_radius_m"].*1e6.*sin.(theta);
            color="#555555",linestyle="--",linewidth=0.8)
        ax.set_title("Mode $m: "*mode_names[m]);ax.set_xlabel("x (μm)");ax.set_ylabel("y (μm)")
        ax.set_aspect("equal")
        fig.colorbar(mesh;ax,shrink=0.75,label="Relative field amplitude")
    end
    fig.suptitle("Fixed spatial modes; dashed circle marks the fiber core")
    finish_figure(fig,joinpath(run_dir,"spatial_modes"))
end
npzwrite(joinpath(run_dir,"spatial_mode_fields.npz"),Dict("x_m"=>x,"y_m"=>y,"modes_y_x_m"=>modes))
modal_rows=NamedTuple[]
for P in power_cases_W
    fig,axes=subplots(1,2;figsize=(12.5,4.5),constrained_layout=true,sharey=true)
    for (case_index,(label,row)) in enumerate((("Baseline",validations[P].base_fine),("Selected",validations[P].best_fine)))
        saved=field_cache[row.id]
        for (j,a) in enumerate((saved.input,saved.linear,saved.nonlinear))
            energy=vec(sum(abs2.(a);dims=(1,3)));fractions=energy./sum(energy)
            axes[case_index].bar(collect(1:3).+(j-2)*0.23,100fractions;
                width=0.23,color=curve_colors[j],label=curve_labels[j])
            for m in 1:3
                push!(modal_rows,(;power_W=P,case=lowercase(label),phase_2_rad=row.phase_2_rad,phase_3_rad=row.phase_3_rad,
                    branch=curve_labels[j],mode=m,energy_fraction=fractions[m]))
            end
        end
        axes[case_index].set_xticks(1:3,collect(mode_names))
        axes[case_index].set_ylim(0,105);axes[case_index].set_ylabel("Fraction of total pulse energy (%)")
        axes[case_index].set_title(label*" "*coeff_label(coefficients(row))*"\nphi="*phase_label(phase_values(row));fontsize=10)
        axes[case_index].legend(;fontsize=8);tidy_axis(axes[case_index])
    end
    fig.suptitle(@sprintf("%.3f kW: modal energy fractions",P/1000))
    finish_figure(fig,joinpath(case_dir(P),"modal_energy_fractions"))
end
save_rows(joinpath(run_dir,"modal_energy_fractions.csv"),modal_rows)

## 11. Reconstruct the coherent beam and individual output modes

At each branch's total pulse peak, add its **complex modal fields** before squaring. The spatial integral is checked against summed modal peak power. Individual-mode maps use each mode's own peak time, so they cannot be added to recover the coherent whole-beam map.

Every physical intensity plot reads the same absolute W/μm² limits and colormap from `plot_settings.toml` beside this notebook. Signed basis-mode illustrations retain their separate −1 to +1 field-amplitude scale.

In [ ]:
beam_maps=Dict{Tuple{Float64,String},Any}()
individual_maps=NamedTuple[]
for P in power_cases_W, (case_index,(label,row)) in enumerate((("baseline",validations[P].base_fine),("selected",validations[P].best_fine)))
    saved=field_cache[row.id]
    beam_maps[(P,label)]=[peak_beam_map(a) for a in (saved.input,saved.linear,saved.nonlinear)]
    fractions=[row.mode1_energy_fraction,row.mode2_energy_fraction,row.mode3_energy_fraction]
    for m in 1:3
        power_t=vec(sum(abs2.(saved.nonlinear[:,m,:]);dims=2));k=argmax(power_t)
        intensity=abs2.(modes[:,:,m]).*power_t[k]./1e12
        push!(individual_maps,(;power_W=P,case_index,label,mode=m,intensity,
            peak_power_W=power_t[k],time_ps=times_ps[k],energy_fraction=fractions[m]))
    end
end
observed_max=max(maximum(maximum(m.intensity) for maps in values(beam_maps) for m in maps),
    maximum(maximum(m.intensity) for m in individual_maps))
write_toml(joinpath(run_dir,"beam_plot_settings.toml"),Dict(
    "shared_min_W_per_um2"=>beam_color_min,"shared_max_W_per_um2"=>beam_color_max,
    "observed_max_W_per_um2"=>observed_max,"source_settings"=>plot_settings_path,"colormap"=>beam_colormap))
for P in power_cases_W
    fig,axes=subplots(2,3;figsize=(15,8.6),constrained_layout=true)
    meshes=[]
    for (i,(label,row)) in enumerate((("baseline",validations[P].base_fine),("selected",validations[P].best_fine)))
        maps=beam_maps[(P,label)]
        for j in 1:3
            ax=axes[i,j];m=maps[j]
            mesh=ax.pcolormesh(x.*1e6,y.*1e6,m.intensity;cmap=beam_colormap,vmin=beam_color_min,vmax=beam_color_max,
                shading="auto",rasterized=true)
            push!(meshes,mesh)
            ax.set_title(@sprintf("%s: %s\npeak %.2f kW at %.3f ps",label,curve_labels[j],m.peak_power_W/1000,m.peak_time_ps);fontsize=10)
            ax.set_xlabel("x (μm)");ax.set_ylabel("y (μm)");ax.set_aspect("equal")
        end
        npzwrite(joinpath(case_dir(P),label*"_beam_maps.npz"),Dict(
            "x_um"=>x.*1e6,"y_um"=>y.*1e6,
            "input_W_per_um2"=>maps[1].intensity,"linear_W_per_um2"=>maps[2].intensity,
            "nonlinear_W_per_um2"=>maps[3].intensity,
            "peak_times_ps"=>[m.peak_time_ps for m in maps],"peak_powers_W"=>[m.peak_power_W for m in maps],
            "shared_color_max_W_per_um2"=>beam_color_max,"launch_coefficients"=>coefficients(row),
            "launch_phases_rad"=>phase_values(row),"launch_complex_coefficients"=>coefficients(row).*cis.(phase_values(row)),
            "normalized_modal_weights"=>coefficients(row).*cis.(phase_values(row))./norm(coefficients(row))))
    end
    fig.colorbar(meshes[end];ax=vec(axes),shrink=0.8,ticks=beam_color_ticks,
        label="Intensity at each pulse's peak (W/μm²)")
    fig.suptitle(@sprintf("%.3f kW: coherent whole beam | start %s\nselected c=%s | phi=%s",
        P/1000,coeff_label(start_coefficients),coeff_label(coefficients(validations[P].best_fine)),
        phase_label(phase_values(validations[P].best_fine)));fontsize=11)
    finish_figure(fig,joinpath(case_dir(P),"baseline_selected_beams"))
end
println("Spatial integrals match summed modal peak powers for all displayed whole-beam maps.")
@printf("Largest plotted physical intensity: %.9g W/μm²\n",observed_max)

In [ ]:
for P in power_cases_W
    fig,axes=subplots(2,3;figsize=(15,8.6),constrained_layout=true)
    meshes=[]
    for m in filter(r->r.power_W==P,individual_maps)
        ax=axes[m.case_index,m.mode]
        mesh=ax.pcolormesh(x.*1e6,y.*1e6,m.intensity;cmap=beam_colormap,vmin=beam_color_min,vmax=beam_color_max,
            shading="auto",rasterized=true)
        push!(meshes,mesh)
        ax.set_title(@sprintf("%s: %s\nenergy %.3g%% | peak %.3g kW at %.3f ps",
            m.label,mode_names[m.mode],100m.energy_fraction,m.peak_power_W/1000,m.time_ps);fontsize=10)
        ax.set_xlabel("x (μm)");ax.set_ylabel("y (μm)");ax.set_aspect("equal")
    end
    fig.colorbar(meshes[end];ax=vec(axes),shrink=0.8,ticks=beam_color_ticks,
        label="Single-mode peak intensity (W/μm²)")
    fig.suptitle(@sprintf("%.3f kW: individual nonlinear output modes",P/1000))
    finish_figure(fig,joinpath(case_dir(P),"baseline_selected_output_modes"))
end

## 12. Report the bounded result

The selected launch is the last accepted feasible descent point. A smaller fine-grid LP01 D_IO is reported as an improvement; budget exhaustion or a failed line search is reported as the actual stop reason. Fine-grid stationarity and global optimality are not established by this single-start run.

CSV files store every evaluation, accepted step, and rejected trial. TOML files store settings, essential checks, and stop reasons. NPZ files and displayed figures retain the actual baseline/selected complex fields, temporal traces, wavelength spectra, and physical beam maps. No simulation or diagnostic sweep is added to improve the reported outcome.

In [ ]:
for P in power_cases_W
    s=summaries[P]
    println("\n",power_label(P))
    println("  coefficients: ",coeff_label(s["selected_coefficients"]))
    println("  relative phases: ",phase_label(s["selected_phases_rad"]),"; phi1 is the reference")
    fractions=s["selected_input_power_fractions"]
    @printf("  input power fractions: [%.3f%%, %.3f%%, %.3f%%]\n",100fractions[1],100fractions[2],100fractions[3])
    @printf("  LP01 fine D_IO: %.8f -> %.8f (%.2f%% reduction)\n",
        s["baseline_D_IO_fine"],s["selected_D_IO_fine"],100s["fractional_D_IO_reduction"])
    println("  stop: ",s["stop_reason"],"; accepted steps: ",s["accepted_steps"])
    println("  essential checks passed: ",s["essential_checks_passed"],
        "; fine-grid improvement: ",s["fine_grid_improvement"])
    println("  fine-grid stationarity not checked; global minimum not established")
    @printf("  max D_IO difference, 5 mm versus 1 mm: %.3g (tolerance %.3g)\n",
        s["max_step_size_D_IO_difference"],dio_tolerance)
    @printf("  LP01 brightness ratio: %.6f; new evaluations: optimization %d/%d; fine %d/%d\n",
        s["brightness_ratio"],used[P][:optimization],max_optimizer_evaluations,used[P][:validation],max_check_evaluations)
end
println("\nResults: ",run_dir)